# アウトバウンド認証

> オリジナルと内容ごと変えています。

アウトバウンド認証により、エージェントと AgentCore Gateway は、インバウンド認証中に認証および承認されたユーザーに代わって、AWS リソースやサードパーティサービスに安全にアクセスできます。AWS リソースやサードパーティサービスとの認証を統合するには、インバウンド認証とアウトバウンド認証の両方を設定する必要があります。

AgentCore Identity がサポートする必要最小限のアクセスと安全な権限委任により、エージェントは AWS リソースや GitHub、Google、Salesforce、Slack などのサードパーティツールにシームレスかつ安全にアクセスできます。エージェントは、事前に承認されたユーザーの同意がある場合、ユーザーに代わって、または独立してこれらのサービスでアクションを実行できます。さらに、安全なトークンボールトを使用して同意疲れを軽減し、合理化された AI エージェントエクスペリエンスを作成できます。

## アウトバウンド認証の設定

まず、サードパーティプロバイダーにクライアントアプリケーションを登録し、次にアウトバウンド認証を作成します。AWS リソースやサードパーティサービス、または AgentCore Gateway ターゲットへのアクセスを検証する方法を指定します。OAuth 2LO/3LO または API キーを使用できます。OAuth では、AgentCore Identity が提供するプロバイダーから選択できます。その場合、AgentCore Identity からプロバイダーの設定詳細を入力します。あるいは、カスタムプロバイダーの詳細を提供することもできます。

ユーザーが AWS リソースやサードパーティサービス、または AgentCore Gateway ターゲットへのアクセスを希望する場合、アウトバウンド認証は、インカミング認証によって提供されたアクセストークンが有効であることを確認し、有効であればリソースへのアクセスを許可します。

<div style="text-align:center">
    <img src="images/outbound_auth.png" width="90%"/>
</div>

## リソース認証情報プロバイダー

これは、エージェントコードがダウンストリームリソースサーバー（Google、GitHub など）の認証情報を取得して、Gmail からメールを取得したり、Google カレンダーに会議を追加したりするなどのアクセスに使用するコンポーネントです。エンドユーザー、エージェントコード、外部認証サーバー間での 2LO および 3LO OAuth2 オーケストレーションフローの実装に関するエージェント開発者の負担を軽減します。AgentCore は、カスタム OAuth2 認証情報プロバイダーと、認証サーバーエンドポイントとプロバイダー固有のパラメーターが事前に入力された Google、GitHub、Slack、Salesforce などの組み込みプロバイダーのリストの両方を提供します。

Bedrock AgentCore Identity は、エージェント開発者が OAuth2 または API キーをサポートする外部リソースで認証するための OAuth2 および API キー認証情報プロバイダーを提供します。以下の例では、API キー認証情報プロバイダーの設定手順を説明します。エージェントは API キー認証情報プロバイダーを使用して、任意のエージェント操作の API キーを取得できます。他の認証情報プロバイダーについてはドキュメントを参照してください。

### リソース認証情報プロバイダーの作成

以下は API キーリソース認証情報プロバイダーを作成する例です。

```
from bedrock_agentcore.services.identity import IdentityClient
identity_client = IdentityClient(region="us-west-2")

api_key_provider = identity_client.create_api_key_credential_provider({
    "name": "APIKey-provider-name",
    "apiKey": "<my-api-key>" # 外部アプリケーションベンダー（OpenAI など）から取得した API キーに置き換えてください
})
print(api_key_provider)
```

### リソース認証情報プロバイダーからのアクセストークンまたは API キーの取得

以下は API キー認証情報プロバイダーから API キーを取得する例です。エージェントは API キーを使用して、LLM や API キー設定を使用する他のサービスと対話できます。認証情報プロバイダーからアクセストークンや API キーなどの認証情報を取得するには、以下のように関数をデコレートできます。

```
import asyncio
from bedrock_agentcore.identity.auth import requires_access_token, requires_api_key

@requires_api_key(
    provider_name="APIKey-provider" # 自分の認証情報プロバイダー名に置き換えてください
)
async def need_api_key(*, api_key: str):
    print(f'received api key for async func: {api_key}')

await need_api_key(api_key="")
```

@require_access_token デコレーターで使用できる様々なパラメーターは以下の通りです。

| パラメーター名 | 説明 |
|:--------------|:-----|
| provider_name | 認証情報プロバイダー名 |
| into | トークンを注入するパラメーター名 |
| scopes | リクエストする OAuth2 スコープ |
| on_auth_url | 認証 URL を処理するためのコールバック |
| auth_flow | 認証フロータイプ（"M2M" または "USER_FEDERATION"） |
| callback_url | OAuth2 コールバック URL |
| force_authentication | 再認証を強制する |
| token_poller | カスタムトークンポーラーの実装 |

# Amazon Bedrock AgentCore ランタイムでの OpenAI モデルを使用した Strands エージェントのホスティング

## 概要

このチュートリアルでは、01-AgentCore-runtime でデプロイした openai-model を使用するエージェントを変更し、API キー認証情報プロバイダーを使用してアウトバウンド認証を設定します。open-ai キーを保存するための API キー認証情報プロバイダーを設定し、このキーを使用するようにエージェントコードを変更します。

### チュートリアルのアーキテクチャ

<div style="text-align:center">
    <img src="images/outbound_auth_api.png" width="90%"/>
</div>

### チュートリアルの詳細

| 情報 | 詳細 |
|:-----|:-----|
| チュートリアルタイプ | 会話型 |
| エージェントタイプ | シングル |
| エージェンティックフレームワーク | Strands エージェント |
| LLM モデル | GPT 4.1 mini |
| チュートリアルコンポーネント | AgentCore ランタイムでのエージェントのホスティング。Strands エージェントと OpenAI モデルの使用 |
| チュートリアルの業種 | 業種横断的 |
| 例の複雑さ | 簡単 |
| 使用する SDK | Amazon BedrockAgentCore Python SDK と boto3 |
| 認証情報プロバイダー | タイプ：API キー |

### チュートリアルの主な機能

* Amazon Bedrock AgentCore ランタイムでのエージェントのホスティング
* OpenAI モデルの使用
* Strands エージェントの使用
* API キー認証情報プロバイダーを使用した AgentCore 出力認証の使用

## 前提条件

このチュートリアルを実行するには以下が必要です：
* Python 3.10+
* AWS 認証情報
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker が稼働していること

In [ ]:
#!uv add -r requirements.txt --active
import os

os.environ['AWS_PROFILE'] = 'cline2'

## エージェントの作成とローカルでの実験

AgentCore Runtime にエージェントをデプロイする前に、実験目的でローカルで開発・実行してみましょう。

本番環境のエージェントアプリケーションでは、エージェント作成プロセスとエージェント呼び出しプロセスを分離する必要があります。AgentCore Runtime では、エージェントの呼び出し部分を `@app.entrypoint` デコレータで装飾し、ランタイムのエントリーポイントとして設定します。まずは実験段階での各エージェントの開発方法を見ていきましょう。

ここでのアーキテクチャは以下のようになります：

<div style="text-align:left">
    <img src="images/architecture_local.png" width="50%"/>
</div>

In [ ]:
%%writefile strands_agents_openai.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool

Japanese translation:

# 計算機ツールをインポートする
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os

##以下の設定を、あなたの Azure API キーの詳細で更新してください。
os.environ["AZURE_API_KEY"] = "<YOUR_API_KEY>"
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

# カスタムツールの作成
@tool
def weather():
    """ 天気を取得する """ # ダミー実装
    return "sunny"

model = "azure/gpt-4.1-mini"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
)


agent = Agent(
    model=litellm_model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

def strands_agent_open_ai(payload):
    """
    ペイロードを使用してエージェントを呼び出す
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = strands_agent_open_ai(json.loads(args.payload))
    print(response)

#### ローカル エージェントの呼び出し

In [ ]:
!python strands_agents_openai.py '{"prompt": "What is the weather now?"}'

## リソース認証情報プロバイダーの作成

In [ ]:
from bedrock_agentcore.services.identity import IdentityClient

from boto3.session import Session
import boto3
boto_session = Session()
region = boto_session.region_name

# API キー プロバイダーの構成
identity_client = IdentityClient(region=region)

api_key_provider = identity_client.create_api_key_credential_provider({
    "name": "openai-apikey-provider",
    "apiKey": "<YOUR_API_KEY>" # 外部アプリケーションベンダー（例えば OpenAI）から取得した API キーに置き換えてください
})
print(api_key_provider)


# AgentCore Runtime へのエージェントのデプロイ準備とリソース認証情報プロバイダーの使用

AgentCore Runtime にエージェントをデプロイしましょう。そのためには以下が必要です：
* `from bedrock_agentcore.runtime import BedrockAgentCoreApp` で Runtime App をインポートする
* コード内で `app = BedrockAgentCoreApp()` を使って App を初期化する
* 呼び出し関数に `@app.entrypoint` デコレータを付ける
* `app.run()` で AgentCoreRuntime にエージェントの実行を制御させる
* 前のステップで作成したリソース認証情報プロバイダーから openAI キーを取得する

### OpenAI モデルを使用した Strands エージェント
GPT 4.1 mini モデルを使用した Strands エージェントから始めましょう。他のすべてのエージェントも全く同じように動作します。

In [ ]:
%%writefile strands_agents_openai.py
import asyncio
from bedrock_agentcore.identity.auth import requires_access_token, requires_api_key
from strands import Agent, tool
from strands_tools import calculator 
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os
from bedrock_agentcore.runtime import BedrockAgentCoreApp

AZURE_API_KEY_FROM_CREDS_PROVIDER = ""


@requires_api_key(
    provider_name="openai-apikey-provider" # 独自の認証情報プロバイダー名に置き換えてください
)
async def need_api_key(*, api_key: str):
    global AZURE_API_KEY_FROM_CREDS_PROVIDER
    print(f'received api key for async func: {api_key}')
    AZURE_API_KEY_FROM_CREDS_PROVIDER = api_key

# モジュールレベルで空の値を出力しないでください - エントリーポイント関数内で出力します

app = BedrockAgentCoreApp()

# API キーはエントリーポイント関数で動的に設定されます
#Azure API Key の詳細で以下の設定を更新してください。
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

# カスタムツールの作成
@tool
def weather():
    """ 天気を取得 """ # ダミー実装
    return "sunny"

# グローバル エージェント変数
agent = None

@app.entrypoint
async def strands_agent_open_ai(payload):
    """
    ペイロードを使用してエージェントを呼び出す
    """
    global AZURE_API_KEY_FROM_CREDS_PROVIDER, agent
    
    print(f"Entrypoint called with AZURE_API_KEY_FROM_CREDS_PROVIDER: '{AZURE_API_KEY_FROM_CREDS_PROVIDER}'")
    
    # API キーがまだ取得されていない場合は取得する
    if not AZURE_API_KEY_FROM_CREDS_PROVIDER:
        print("Attempting to retrieve API key...")
        try:
            await need_api_key(api_key="")
            print(f"API key retrieved: '{AZURE_API_KEY_FROM_CREDS_PROVIDER}'")
            os.environ["AZURE_API_KEY"] = AZURE_API_KEY_FROM_CREDS_PROVIDER
            print("Environment variable AZURE_API_KEY set")
        except Exception as e:
            print(f"Error retrieving API key: {e}")
            raise
    else:
        print("API key already available")
    
    # API キーが設定された後にエージェントを初期化する
    if agent is None:
        print("Initializing agent with API key...")
        model = "azure/gpt-4.1-mini"
        litellm_model = LiteLLMModel(
            model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
        )
        
        agent = Agent(
            model=litellm_model,
            tools=[calculator, weather],
            system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
        )
        print("Agent initialized successfully")
    
    user_input = payload.get("prompt")
    print(f"User input: {user_input}")
    
    try:
        response = agent(user_input)
        print(f"Agent response: {response}")
        return response.message['content'][0]['text']
    except Exception as e:
        print(f"Error in agent processing: {e}")
        raise

if __name__ == "__main__":
    app.run()


## バックグラウンドで何が起こっているのか？

`BedrockAgentCoreApp` を使用すると、自動的に以下のことが行われます：

* ポート 8080 でリクエストを待ち受ける HTTP サーバーを作成
* エージェントの要件を処理するために必要な `/invocations` エンドポイントを実装
* ヘルスチェック用の `/ping` エンドポイントを実装（非同期エージェントにとって非常に重要）
* 適切なコンテンツタイプとレスポンス形式を処理
* AWS 標準に従ったエラー処理を管理

## AgentCore Runtime へのエージェントのデプロイ

`CreateAgentRuntime` 操作は包括的な構成オプションをサポートしており、コンテナイメージ、環境変数、暗号化設定を指定することができます。また、プロトコル設定 (HTTP、MCP) や認証メカニズムを構成して、クライアントがエージェントとどのように通信するかを制御することもできます。

**注意:** 運用のベストプラクティスは、コードをコンテナとしてパッケージ化し、CI/CD パイプラインと IaC を使用して ECR にプッシュすることです

このチュートリアルでは、Amazon Bedrock AgentCode Python SDK を使用して、アーティファクトを簡単にパッケージ化し、AgentCore ランタイムにデプロイします。

### ランタイムロールの作成

始める前に、AgentCore ランタイム用の IAM ロールを作成しましょう。これは、あなたのために事前に開発されたユーティリティ関数を使用して行います。

In [ ]:
import sys
import os

# 現在のノートブックのディレクトリを取得する
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

# utils.py の場所まで上に移動する
utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# sys.path に追加する
sys.path.insert(0, utils_dir)

from utils import create_agentcore_role

agent_name="strands_agents_openai"
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)

### AgentCore Runtime デプロイメントの構成

次に、スターターツールキットを使用して、エントリーポイント、先ほど作成した実行ロール、および要件ファイルを含む AgentCore Runtime デプロイメントを構成します。また、起動時に Amazon ECR リポジトリを自動作成するようにスターターキットを構成します。

構成ステップでは、アプリケーションコードに基づいて Docker ファイルが生成されます

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import boto3
import json
boto_session = Session()
region = boto_session.region_name
region

agentcore_runtime = Runtime()
agent_name="strands_agents_openai"

response = agentcore_runtime.configure(
    entrypoint="strands_agents_openai.py",
    execution_role=agentcore_iam_role['Role']['Arn'],
    auto_create_ecr=True,
    agent_name=agent_name,
    requirements_file="requirements.txt",
    region=region
)
response

### AgentCore Runtime へのエージェントの起動

Docker ファイルができたので、エージェントを AgentCore Runtime に起動しましょう。これにより Amazon ECR リポジトリと AgentCore Runtime が作成されます。

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

### AgentCore ランタイムのステータス確認
AgentCore ランタイムをデプロイしたので、そのデプロイステータスを確認しましょう

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### AgentCore ランタイムの呼び出し

最後に、ペイロードを使用して AgentCore ランタイムを呼び出すことができます

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "Hello"}, user_id="userid_1234567890")
invoke_response

### 呼び出し結果の処理

呼び出し結果をアプリケーションに含めるために処理することができます

In [ ]:
from IPython.display import Markdown, display
response_text = json.loads(invoke_response['response'][0].decode("utf-8"))
display(Markdown(response_text))

### AgentCore Runtime を boto3 で呼び出す

AgentCore Runtime が作成されたので、任意の AWS SDK でそれを呼び出すことができます。例えば、boto3 の `invoke_agent_runtime` メソッドを使用することができます。

In [ ]:
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    runtimeUserId="userid_1234567890",
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How much is 2X2?"})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                logger.info(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## クリーンアップ (任意)

作成した AgentCore Runtime をクリーンアップしましょう

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

iam_client = boto3.client('iam')

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

policies = iam_client.list_role_policies(
    RoleName=agentcore_iam_role['Role']['RoleName'],
    MaxItems=100
)

for policy_name in policies['PolicyNames']:
    iam_client.delete_role_policy(
        RoleName=agentcore_role_name,
        PolicyName=policy_name
    )
iam_response = iam_client.delete_role(
    RoleName=agentcore_role_name
)

# おめでとうございます！